# Fase 18: Extração das métricas dos áudios
Esse notebook extrai métricas dos áudios


# Análise de Arquivos de Áudio OGG e Comparação de Ferramentas Speech-to-Text

## Instalação de Dependências

In [2]:
!pip install librosa soundfile matplotlib numpy pandas plotly scipy vosk transformers pyaudio torch torchaudio accelerate hf_xet peft numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 7.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 48.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 45.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 51.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Vendo quantos áudios foram baixados

In [10]:
import os
from collections import Counter

# Conta quantos arquivos terminam com .mp3 na pasta
qtd_audios = len([arquivo for arquivo in os.listdir("audios") if arquivo.endswith(".mp3")])

print(f"Você já tem {qtd_audios} áudios baixados!")

audio_folder = "audios" 

ids_na_pasta = [os.path.splitext(f)[0] for f in os.listdir(audio_folder) if os.path.isfile(os.path.join(audio_folder, f))]
contagem = Counter(ids_na_pasta)

repetidos = {id_video: qtd for id_video, qtd in contagem.items() if qtd > 1}

if repetidos:
    print(f"Atenção! Encontrei {len(repetidos)} IDs repetidos na pasta:")
    for id_video, qtd in repetidos.items():
        print(f" - ID: {id_video} (aparece {qtd} vezes)")
else:
    print("Tudo limpo! Cada ID aparece apenas uma vez na sua pasta de áudios.")

Você já tem 8973 áudios baixados!
Atenção! Encontrei 1 IDs repetidos na pasta:
 - ID: _uOcCRlVQfU (aparece 2 vezes)


In [ ]:
AUDIO_DIR = "audios"

In [ ]:
import zipfile
import os

os.chdir(AUDIO_DIR)

for zip in os.listdir():
    if zip.endswith('.zip'):
        audio = os.path.splitext(zip)[0]
        destino = os.path.join(AUDIO_DIR,audio)

        os.makedirs(destino, exist_ok=True)
        caminho = os.path.join(AUDIO_DIR,zip)

        with zipfile.ZipFile(caminho,'r') as zip_ref:
            zip_ref.extractall(destino)
        
        print(f"Extraído: {zip} -> {audio}")

In [ ]:
import zipfile
import os
import shutil

for pasta in os.listdir(AUDIO_DIR):
    caminho_pasta = os.path.join(AUDIO_DIR, pasta)

    if os.path.isdir(caminho_pasta):
        for arquivo in os.listdir(caminho_pasta):
            if arquivo.endswith('.mp3'):
                origem = os.path.join(caminho_pasta, arquivo)
                destino = os.path.join(AUDIO_DIR, arquivo)

                shutil.move(origem, destino)
        
        shutil.rmtree(caminho_pasta)
        print(f"Removida pasta: {pasta}")


## 2. Funções para Análise Exploratória

In [ ]:
def list_audio_files(directory, extension='.mp3'):
    """Lista todos os arquivos de áudio com a extensão especificada em um diretório."""
    pattern = os.path.join(directory, f'*{extension}')
    return glob.glob(pattern)

def extract_audio_features(file_path, silence_threshold_db=-35, min_silence_duration=0.3):
    """Extrai características básicas de um arquivo de áudio."""
    try:
        y, sr = librosa.load(file_path, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)

        mean_amplitude = np.mean(np.abs(y))
        max_amplitude = np.max(np.abs(y))
        rms = np.sqrt(np.mean(y ** 2))

        # Análise por quadros
        frame_length = 1024
        hop_length = 512
        rms_frames = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
        db_frames = librosa.amplitude_to_db(rms_frames, ref=np.max)

        # Silêncio
        silent_frames = db_frames < silence_threshold_db
        silence_regions = 0
        in_silence = False
        min_frames = int((min_silence_duration * sr) / hop_length)
        count = 0
        for i in range(len(silent_frames)):
            if silent_frames[i]:
                count += 1
                if not in_silence:
                    in_silence = True
            else:
                if in_silence and count >= min_frames:
                    silence_regions += 1
                count = 0
                in_silence = False

        # SNR estimado
        noise_energy = np.mean(rms_frames[silent_frames]) if np.any(silent_frames) else 1e-10
        signal_energy = np.mean(rms_frames[~silent_frames]) if np.any(~silent_frames) else 1e-10
        snr = 20 * np.log10(signal_energy / noise_energy) if noise_energy > 0 else float("inf")

        # Zero-Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y=y)[0].mean()

        return {
            "arquivo": os.path.basename(file_path),
            "duracao_segundos": duration,
            "frequencia_amostragem": sr,
            "amplitude_media": mean_amplitude,
            "amplitude_maxima": max_amplitude,
            "amplitude_ratio (mean/max)": mean_amplitude/max_amplitude,
            "snr_estimado_db": snr,
            "zero_crossing_rate": zcr,
            "regioes_silencio": silence_regions
        }

    except Exception as e:
        return {"erro": str(e)}

def plot_waveform(y, sr, title="Forma de Onda"):
    """Plota a forma de onda de um áudio."""
    plt.figure(figsize=(12, 4))
    librosa.display.waveshow(y, sr=sr)
    plt.title(title)
    plt.xlabel("Tempo (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

def plot_spectrogram(y, sr, title="Espectrograma"):
    """Plota o espectrograma de um áudio."""
    plt.figure(figsize=(12, 6))
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

def calculate_snr(y, percentile_noise=10, percentile_signal=90):
    """Calcula uma estimativa de SNR usando percentis."""
    noise_level = np.percentile(np.abs(y), percentile_noise)
    signal_level = np.percentile(np.abs(y), percentile_signal)
    if noise_level > 0:
        return 20 * np.log10(signal_level / noise_level)
    else:
        return float('inf')

## Descrição das métricas


---
🎧 1. `arquivo`

**Descrição:**
Nome do arquivo de áudio analisado.

**Utilidade:**
Serve para identificação e rastreamento dos dados no conjunto de amostras.

---

⏱ 2. `duracao_segundos`

**Descrição:**
Duração total do áudio em segundos.

**Interpretação:**

* Áudios muito curtos (ex: < 1s) podem indicar mensagens acidentais ou silenciosas.
* Áudios muito longos (> 30s) podem conter múltiplos assuntos ou dificultar a compreensão rápida.

**Valor típico:**
Geralmente entre 1 e 60 segundos para áudios de WhatsApp.

---

🎼 3. `frequencia_amostragem`

**Descrição:**
Taxa de amostragem do áudio, em Hz (ex: 44.100 Hz).

**Interpretação:**

* Quanto maior a frequência, mais detalhes do som são preservados.
* Frequências típicas:

  * **8000 Hz**: Qualidade de telefonia
  * **16000 Hz - 44100 Hz**: Qualidade intermediária a alta

**Valor recomendado:**
Pelo menos **16000 Hz** para boa inteligibilidade.

---

🔊 4. `amplitude_media`

**Descrição:**
Média dos valores de amplitude do sinal de áudio.

**Interpretação:**

* Valores variam entre **0 e 1** se normalizado.
* Valores baixos (ex: < 0.05) indicam áudio silencioso.
* Valores mais altos indicam presença constante de som.

**Utilidade:**
Ajuda a detectar se o áudio está muito baixo ou mal captado.

---

🔊 5. `amplitude_maxima`

**Descrição:**
Maior valor de amplitude no áudio.

**Interpretação:**

* Varia entre **0 e 1** (em escala normalizada).
* Um valor muito próximo de 1 pode indicar **picos de volume** ou distorção.
* Valores muito baixos (< 0.1) indicam ausência de trechos expressivos ou falha na gravação.

---

📉 6. `amplitude_ratio (mean/max)`

**Descrição:**
Razão entre a amplitude média e a máxima.

**Interpretação:**

* Varia entre **0 e 1**.
* **Próximo de 1**: sinal mais estável, volume constante.
* **Próximo de 0**: sinal com muitos picos e silêncio.

**Utilidade:**
Distingue áudios com fala contínua dos que têm **interrupções**, ruídos ou pausas longas.

---

🔇 7. `snr_estimado_db` (Signal-to-Noise Ratio)

**Descrição:**
Estimativa da razão sinal-ruído em decibéis (dB).

**Interpretação:**

* Valores típicos:

  * **< 10 dB**: Áudio com muito ruído.
  * **10–20 dB**: Qualidade aceitável.
  * **> 20 dB**: Boa qualidade de gravação.
* Quanto **maior o SNR**, melhor a separação entre voz e ruído.

**Utilidade:**
Essencial para avaliar a **qualidade do áudio** e filtragem de ruídos.

---

⚡ 8. `zero_crossing_rate` (ZCR)

**Descrição:**
Taxa de cruzamento por zero: quantas vezes o sinal muda de positivo para negativo por segundo.

**Interpretação:**

* Sons **agudos ou ruidosos** → alta ZCR.
* Sons **graves ou com fala** → baixa ZCR.
* Típicos valores:

  * **Fala:** 0.01 – 0.1
  * **Ruído branco:** > 0.2

**Utilidade:**
Ajuda a distinguir entre fala, silêncio e ruído.

---

🕳 9. `regioes_silencio`

**Descrição:**
Lista de regiões no áudio onde há **silêncio detectado**, com timestamps.

**Interpretação:**

* Pode indicar **pausas naturais na fala**, **hesitação**, ou **problemas técnicos** (microfone cortado).
* Áudios com muitos silêncios curtos podem ter fluência comprometida.
* Longos períodos de silêncio podem sinalizar inatividade ou desinteresse.

**Utilidade:**
Permite:

* Cortar silêncios automaticamente.
* Avaliar fluidez da mensagem.
* Otimizar transcrição ou resumo.

---

✅ Conclusão

Essas métricas permitem avaliar aspectos técnicos e qualitativos dos áudios de WhatsApp, como **clareza**, **ruído**, **intensidade da fala** e **estrutura temporal**. Elas são especialmente úteis em aplicações como:

* **Filtragem de áudios úteis** vs. irrelevantes
* **Transcrição automática com maior precisão**
---


In [12]:
# Listar arquivos de áudio OGG
audio_files = list_audio_files(AUDIO_DIR)
print(f"Encontrados {len(audio_files)} arquivos de áudio MP3")

NameError: name 'list_audio_files' is not defined

In [ ]:
# Extrair características de todos os arquivos
audio_features = []
for file_path in audio_files:
    features = extract_audio_features(file_path)
    if features:
        audio_features.append(features)

# Criar DataFrame com características
audio_df = pd.DataFrame(audio_features)

display(audio_df)



In [ ]:
# Estatísticas descritivas dos arquivos de áudio
audio_stats = audio_df.describe()
print("Estatísticas dos arquivos de áudio:")
display(audio_stats)